# Population Density Feature Merge Pipeline

**Purpose:** Join the annual population density panel (1994–2020) onto the merged
weather feature table, adding `population_density` as a new column keyed by
`(lon, lat, year)`.

**Pipeline Overview:**

| Phase | Steps |
|-------|-------|
| **A. Load & Inspect Inputs** | Load population parquet and merged weather parquet; verify shapes, dtypes, and year alignment |
| **B. Merge & Validate** | Left-join on `(lon, lat, year)`; assert row count unchanged; check missing rates; save output |

**Inputs:**
- `Clean_Data/Population/population_density_yearly_1994_2020.parquet` — annual population density
  *(from `02_03_Data_Clean_-_Population.ipynb`)*
- `Clean_Data/Feature_Data/Weather_Data_Merged.parquet` — merged weather feature table
  *(from `03_01_Merge_Features_-_Weather_Data.ipynb`)*

**Output:**
- `Clean_Data/Feature_Data/Weather_Population_Merged.parquet` — weather + population (127M rows × 17 cols)

> **Join strategy:** `LEFT` join on `(lon, lat, year)` — every weather row is kept.
> Grid cells outside the population domain produce a small number of `NaN` values
> in `population_density`, which is expected (~0.08%).

## 0. Configuration

Centralized path configuration — **edit this cell only** to adapt to your local setup.

In [1]:
import os

# ====================== EDIT THESE PATHS ======================
PROJECT_ROOT = r"E:\zcao\CA_Wildfire"

# Input: annual population panel (from 02_03_Data_Clean_-_Population.ipynb)
POPULATION_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Population",
    "population_density_yearly_1994_2020.parquet"
)

# Input: merged weather features (from 03_01_Merge_Features_-_Weather_Data.ipynb)
WEATHER_MERGED_PATH = os.path.join(
    PROJECT_ROOT, "Clean_Data", "Feature_Data",
    "Weather_Data_Merged.parquet"
)

# Output
FEATURE_DATA_DIR = os.path.join(PROJECT_ROOT, "Clean_Data", "Feature_Data")
OUTPUT_FILE      = "Weather_Population_Merged.parquet"

os.makedirs(FEATURE_DATA_DIR, exist_ok=True)

print(f"Population path     : {POPULATION_PATH}")
print(f"Weather merged path : {WEATHER_MERGED_PATH}")
print(f"Output dir          : {FEATURE_DATA_DIR}")

# Quick existence checks
for label, path in [("POPULATION_PATH",     POPULATION_PATH),
                    ("WEATHER_MERGED_PATH", WEATHER_MERGED_PATH)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {label}")

Population path     : E:\zcao\CA_Wildfire\Clean_Data\Population\population_density_yearly_1994_2020.parquet
Weather merged path : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Data_Merged.parquet
Output dir          : E:\zcao\CA_Wildfire\Clean_Data\Feature_Data
  [OK] POPULATION_PATH
  [OK] WEATHER_MERGED_PATH


## 1. Environment Setup

In [2]:
import sys
import gc
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import pyproj
from datetime import datetime

gc.collect()

print(f"Python : {sys.version.split('|')[0].strip()}")
print(f"pandas : {pd.__version__}")
print(f"numpy  : {np.__version__}")
print(f"pyproj : {pyproj.__version__}")

Python : 3.9.13 (main, Aug 25 2022, 23:51:50) [MSC v.1916 64 bit (AMD64)]
pandas : 2.2.2
numpy  : 1.24.4
pyproj : 3.6.1


---

# Phase A: Load & Inspect Inputs

Load both datasets independently and validate shapes, dtypes, and year coverage
before attempting the join. Confirming year alignment early catches mismatches
that would silently produce `NaN`-filled columns after a left join.

## 2. Load Population Data

The annual population panel covers 1994–2020 at the veg-filtered grid resolution.
Each row represents one `(year, lon, lat)` combination with a census-snapshot
density value (people / km²).

In [3]:
population_dat = pd.read_parquet(POPULATION_PATH)

print(f"Shape   : {population_dat.shape}")
print(f"Columns : {list(population_dat.columns)}")
print(f"dtypes  :\n{population_dat.dtypes.to_string()}")

Shape   : (388638, 4)
Columns : ['year', 'lon', 'lat', 'population_density']
dtypes  :
year                    int64
lon                   float64
lat                   float64
population_density    float64


In [4]:
print(f"Year range   : {population_dat['year'].min()} – {population_dat['year'].max()}")
print(f"Unique years : {sorted(population_dat['year'].unique())}")
print()
missing_pop = population_dat.isnull().mean().mul(100).rename('missing_%')
print("Missing rate (%):\n", missing_pop.to_string())

Year range   : 1994 – 2020
Unique years : [1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

Missing rate (%):
 year                  0.000000
lon                   0.000000
lat                   0.000000
population_density    2.987356


## 3. Load Merged Weather Feature Table

Load the wide weather feature table produced by `03_01`. This is the largest
dataset in the pipeline (~127M rows). Confirm shape, dtypes, and date range.

In [5]:
all_features = pd.read_parquet(WEATHER_MERGED_PATH)

print(f"Shape   : {all_features.shape}")
print(f"\nColumn dtypes:")
print(all_features.dtypes.to_string())

Shape   : (127478960, 16)

Column dtypes:
day                                          datetime64[ns]
lat                                                 float64
lon                                                 float64
SWE                                                 float32
year                                                  int32
dead_fuel_moisture_1000hr                           float64
dead_fuel_moisture_100hr                            float64
max_air_temperature                                 float64
max_relative_humidity                               float64
min_air_temperature                                 float64
min_relative_humidity                               float64
precipitation_amount                                float64
specific_humidity                                   float64
surface_downwelling_shortwave_flux_in_air           float64
wind_from_direction                                 float32
wind_speed                                          float6

In [6]:
print(f"Date range   : {all_features['day'].min().date()} → {all_features['day'].max().date()}")
print(f"Unique years : {sorted(all_features['year'].unique())}")

Date range   : 1994-01-01 → 2020-09-30
Unique years : [1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]


## 4. Year Alignment Check

Assert that both datasets share exactly the same set of years.
A mismatch would silently produce `NaN`-filled rows after the left join.

In [7]:
pop_years     = set(population_dat['year'].unique())
weather_years = set(all_features['year'].unique())

only_in_pop     = pop_years - weather_years
only_in_weather = weather_years - pop_years

if only_in_pop:
    print(f"WARNING — years in population only : {sorted(only_in_pop)}")
if only_in_weather:
    print(f"WARNING — years in weather only    : {sorted(only_in_weather)}")

assert pop_years == weather_years, \
    f"Year mismatch — pop only: {only_in_pop} | weather only: {only_in_weather}"

print(f"Year alignment OK — {len(pop_years)} years matched.")

Year alignment OK — 27 years matched.


---

# Phase B: Merge & Validate

Left-join the population panel onto the weather feature table on `(lon, lat, year)`.
Population density is constant within a year, so each `(lon, lat, year)` key in
the weather table picks up the same scalar value regardless of the day.

## 5. Left-Join Population onto Weather Features

A `LEFT` join preserves all 127M weather rows. The row count must be identical
before and after — any increase would indicate duplicate keys in the population
table; any decrease would indicate an accidental inner join.

In [8]:
rows_before = len(all_features)
print(f"Before merge : {rows_before:,} rows × {all_features.shape[1]} cols")

all_features = pd.merge(
    all_features, population_dat,
    on=['lon', 'lat', 'year'],
    how='left'
)

rows_after = len(all_features)
print(f"After merge  : {rows_after:,} rows × {all_features.shape[1]} cols")

assert rows_before == rows_after, \
    f"Row count changed after left join: {rows_before:,} -> {rows_after:,}"
print("Row count unchanged — left join OK.")

Before merge : 127,478,960 rows × 16 cols
After merge  : 127,478,960 rows × 17 cols
Row count unchanged — left join OK.


## 6. Post-Merge Validation

Confirm the schema is correct (17 columns) and review missing rates.
Key expectations:
- `population_density` should have < 1% missing (subregion-boundary cells)
- All weather variable missing rates should be unchanged from Phase A

In [9]:
print(f"Shape   : {all_features.shape}")
print(f"Columns : {list(all_features.columns)}")

Shape   : (127478960, 17)
Columns : ['day', 'lat', 'lon', 'SWE', 'year', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr', 'max_air_temperature', 'max_relative_humidity', 'min_air_temperature', 'min_relative_humidity', 'precipitation_amount', 'specific_humidity', 'surface_downwelling_shortwave_flux_in_air', 'wind_from_direction', 'wind_speed', 'population_density']


In [10]:
missing_rates = all_features.isnull().mean().mul(100).rename('missing_%')
print("Missing rate per column (%):\n")
print(missing_rates.to_string())

pop_missing = missing_rates['population_density']
print(f"\npopulation_density missing: {pop_missing:.4f}%")
if pop_missing > 1.0:
    print("  NOTE: >1% missing — check subregion/population grid overlap.")
else:
    print("  OK — within expected range.")

Missing rate per column (%):

day                                           0.000000
lat                                           0.000000
lon                                           0.000000
SWE                                           1.709074
year                                          0.000000
dead_fuel_moisture_1000hr                     0.167830
dead_fuel_moisture_100hr                      0.167830
max_air_temperature                           0.116906
max_relative_humidity                         0.167830
min_air_temperature                           0.116906
min_relative_humidity                         0.167832
precipitation_amount                         57.369750
specific_humidity                             0.167830
surface_downwelling_shortwave_flux_in_air     0.167830
wind_from_direction                           0.267880
wind_speed                                    0.167830
population_density                            0.076640

population_density missing: 0.0766

## 7. Save Output

Write the merged feature table to Parquet. This file is the input for the
next merge step (LAI, subregion labels, and vegetation features).

In [11]:
output_path = os.path.join(FEATURE_DATA_DIR, OUTPUT_FILE)

all_features.to_parquet(output_path, index=False)

print(f"Saved -> {output_path}")
print(f"File size  : {os.path.getsize(output_path) / 1e9:.2f} GB")
print(f"Final shape: {all_features.shape[0]:,} rows × {all_features.shape[1]} cols")
print(f"Columns    : {list(all_features.columns)}")

del all_features, population_dat
gc.collect()

Saved -> E:\zcao\CA_Wildfire\Clean_Data\Feature_Data\Weather_Population_Merged.parquet
File size  : 1.59 GB
Final shape: 127,478,960 rows × 17 cols
Columns    : ['day', 'lat', 'lon', 'SWE', 'year', 'dead_fuel_moisture_1000hr', 'dead_fuel_moisture_100hr', 'max_air_temperature', 'max_relative_humidity', 'min_air_temperature', 'min_relative_humidity', 'precipitation_amount', 'specific_humidity', 'surface_downwelling_shortwave_flux_in_air', 'wind_from_direction', 'wind_speed', 'population_density']


0

## 8. Summary

| Phase | Step | Description | Key Result |
|-------|------|-------------|------------|
| A | Load Population | `population_density_yearly_1994_2020.parquet` | 388k rows × 4 cols |
| A | Load Weather | `Weather_Data_Merged.parquet` | 127M rows × 16 cols |
| A | Year Check | Assert identical year sets in both tables | 1994–2020 aligned |
| B | Left Join | Merge on `(lon, lat, year)` | Row count unchanged |
| B | Validate | Shape (17 cols) + missing rates | `population_density` ~0.08% NaN |
| B | Save | `Weather_Population_Merged.parquet` | 127M rows × 17 cols |